<a href="https://colab.research.google.com/github/vaishaliojha08-sys/FirstModel/blob/main/Regularization%26HyperparameterTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.metrics import r2_score
import time
from sklearn.preprocessing import StandardScaler


In [ ]:
#Generate synthetic house price dataset
np.random.seed(42)
X, y = make_regression(n_samples=1000,n_features=20,n_informative=15,noise=20,random_state=42)

#Add feature names for better interpretation
feature_names = [f"feature_{i+1}" for i in range(20)]

#split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
print("Part A: Compare Ridge, Lasso, ElasticNet at Alpha:1.0")

#Baseline: Linear Regression (no regularization)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

lr_train_r2 = r2_score(y_train, lr.predict(X_train_scaled))
lr_test_r2 = r2_score(y_test, lr.predict(X_test_scaled))
lr_gap = lr_train_r2 - lr_test_r2

print("Baseline (No Regularization)")
print(f"Train R2:{lr_train_r2:.4f}")
print(f"Test R2:{lr_test_r2:.4f}")
print(f"Overfitting Gap:{lr_gap:.4f} ({lr_gap*100:.2f})%")

Part A: Compare Ridge, Lasso, ElasticNet at Alpha:1.0
Baseline (No Regularization)
Train R2:0.9926
Test R2:0.9922
Overfitting Gap:0.0004 (0.04)%


In [ ]:
#Ridge Regression L2

ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_scaled, y_train)

ridge_train_r2 = r2_score(y_train, ridge.predict(X_train_scaled))
ridge_test_r2 = r2_score(y_test, ridge.predict(X_test_scaled))
ridge_gap = ridge_train_r2 - ridge_test_r2

print("Ridge Regression L2")
print(f"Train R2:{ridge_train_r2:.4f}")
print(f"Test R2:{ridge_test_r2:.4f}")
print(f"Overfitting Gap:{ridge_gap:.4f} ({ridge_gap*100:.2f})%")


Ridge Regression L2
Train R2:0.9926
Test R2:0.9922
Overfitting Gap:0.0004 (0.04)%


In [ ]:
#Lasso Regression L1

lasso = Lasso(alpha=1.0, max_iter=10000, random_state=42)
lasso.fit(X_train_scaled, y_train)

lasso_train_r2 = r2_score(y_train, lasso.predict(X_train_scaled))
lasso_test_r2 = r2_score(y_test, lasso.predict(X_test_scaled))
lasso_gap = lasso_train_r2 - lasso_test_r2

print("Ridge Regression L2")
print(f"Train R2:{lasso_train_r2:.4f}")
print(f"Test R2:{lasso_test_r2:.4f}")
print(f"Overfitting Gap:{lasso_gap:.4f} ({lasso_gap*100:.2f})%")


Ridge Regression L2
Train R2:0.9923
Test R2:0.9918
Overfitting Gap:0.0005 (0.05)%


In [ ]:
#ElasticNet L1 + L2

elastic = ElasticNet(alpha=1.0,l1_ratio=0.5,max_iter=10000,random_state=42)
elastic.fit(X_train_scaled,y_train)

elastic_train_r2 = r2_score(y_train, elastic.predict(X_train_scaled))
elastic_test_r2 = r2_score(y_test, elastic.predict(X_test_scaled))
elastic_gap = elastic_train_r2 - elastic_test_r2

print("Ridge Regression L2")
print(f"Train R2:{elastic_train_r2:.4f}")
print(f"Test R2:{elastic_test_r2:.4f}")
print(f"Overfitting Gap:{elastic_gap:.4f} ({elastic_gap*100:.2f})%")

Ridge Regression L2
Train R2:0.8801
Test R2:0.8738
Overfitting Gap:0.0063 (0.63)%


In [ ]:
#Determine which shows least overfitting

gaps = {'Ridge':ridge_gap,
        'Lasso':lasso_gap,
        'Elastic':elastic_gap}
best_model = min(gaps, key=gaps.get)

print("Model with least Overfitting Gap")
print(f"Baseline :{lr_gap:.4f} ({lr_gap*100:.2f})%")
print(f"Ridge :{ridge_gap:.4f} ({ridge_gap*100:.2f})%")
print(f"Lasso :{lasso_gap:.4f} ({lasso_gap*100:.2f})%")
print(f"Elastic Net :{elastic_gap:.4f} ({elastic_gap*100:.2f})%")
print(f"\n {best_model} shows the LEAST Overfitting(gap={gaps[best_model]:4f})")

Model with least Overfitting Gap
Baseline :0.0004 (0.04)%
Ridge :0.0004 (0.04)%
Lasso :0.0005 (0.05)%
Elastic Net :0.0063 (0.63)%

 Ridge shows the LEAST Overfitting(gap=0.000445)


In [ ]:
print(np.logspace(-3,3,6))

[1.00000000e-03 1.58489319e-02 2.51188643e-01 3.98107171e+00
 6.30957344e+01 1.00000000e+03]


In [ ]:
#Grid search for optimal ridge alpha
import time

param_grid = {'alpha':[0.001,0.01,0.1,1,10,100]}

print("Cross Validation: 5 Fold")

start_time = time.time()
grid_search = GridSearchCV(
    Ridge(random_state = 42),
    param_grid = param_grid,
    cv =5,
    scoring = 'r2',
    return_train_score = True,
    n_jobs =-1
)

grid_search.fit(X_train_scaled,y_train)
grid_time = time.time() - start_time

print(f"Grid serach completed in: {grid_time:.2f} seconds")
print(f"Best Alpha: {grid_search.best_params_['alpha']}")
print(f"Best cross-validation score(R2) {grid_search.best_score_:.4f}")

print(f"Detailed Results:")
results_df = pd.DataFrame(grid_search.cv_results_)
for idx, alpha in enumerate(param_grid['alpha']):
  mean_train_score = results_df.loc[idx,'mean_train_score']
  mean_test_score = results_df.loc[idx,'mean_test_score']
  print(f"Alpha is: {alpha:7.3f}, Train R2: {mean_train_score:.4f}, CV R2: {mean_test_score:.4f}, Gap: {mean_train_score - mean_test_score:.4f} ")

#Test the best model

best_ridge = grid_search.best_estimator_
test_score = r2_score(y_test, best_ridge.predict(X_test_scaled))
print(f"\nBest Ridge model for test score R2: {test_score:.4f}")

Cross Validation: 5 Fold
Grid serach completed in: 0.17 seconds
Best Alpha: 1
Best cross-validation score(R2) 0.9922
Detailed Results:
Alpha is:   0.001, Train R2: 0.9927, CV R2: 0.9922, Gap: 0.0005 
Alpha is:   0.010, Train R2: 0.9927, CV R2: 0.9922, Gap: 0.0005 
Alpha is:   0.100, Train R2: 0.9927, CV R2: 0.9922, Gap: 0.0005 
Alpha is:   1.000, Train R2: 0.9927, CV R2: 0.9922, Gap: 0.0005 
Alpha is:  10.000, Train R2: 0.9924, CV R2: 0.9919, Gap: 0.0005 
Alpha is: 100.000, Train R2: 0.9745, CV R2: 0.9731, Gap: 0.0014 

Best Ridge model for test score R2: 0.9922


In [ ]:
#Grid search vs Randomized search for ElasticNet

# Grid Search for Elastic Net
print(f"\n--- GRID SEARCH ---")
param_grid_elastic = {
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0, 0.25, 0.5, 0.75, 1]
}
n_grid_combinations = len(param_grid_elastic['alpha']) * len(param_grid_elastic['l1_ratio'])
print(f"Parameter grid:")
print(f"  alpha: {param_grid_elastic['alpha']}")
print(f"  l1_ratio: {param_grid_elastic['l1_ratio']}")
print(f"Total combinations: {n_grid_combinations}")

start_time = time.time()
grid_elastic = GridSearchCV(
    ElasticNet(max_iter=10000, random_state=42),
    param_grid=param_grid_elastic,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
grid_elastic.fit(X_train_scaled, y_train)
grid_elastic_time = time.time() - start_time

print(f"Execution time: {grid_elastic_time:.2f} seconds")
print(f"Best parameters: {grid_elastic.best_params_}")
print(f"Best CV score: {grid_elastic.best_score_:.4f}")

# Randomized Search for Elastic Net
print(f"\n--- RANDOMIZED SEARCH ---")
from scipy.stats import uniform

param_dist_elastic = {
    'alpha': uniform(0.001, 100),  # Uniform distribution from 0.001 to 100.001
    'l1_ratio': uniform(0, 1)      # Uniform distribution from 0 to 1
}
n_random_iterations = 25
print(f"Parameter distributions:")
print(f"  alpha: uniform(0.001, 100)")
print(f"  l1_ratio: uniform(0, 1)")
print(f"Random iterations: {n_random_iterations}")

start_time = time.time()
random_elastic = RandomizedSearchCV(
    ElasticNet(max_iter=10000, random_state=42),
    param_distributions=param_dist_elastic,
    n_iter=n_random_iterations,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)
random_elastic.fit(X_train_scaled, y_train)
random_elastic_time = time.time() - start_time

print(f"Execution time: {random_elastic_time:.2f} seconds")
print(f"Best parameters: {random_elastic.best_params_}")
print(f"Best CV score: {random_elastic.best_score_:.4f}")

print(f"\n{'='*80}")
print(f"ANSWER (c): Grid Search vs Randomized Search Comparison")
print(f"{'='*80}")

print(f"\n(i) NUMBER OF COMBINATIONS EVALUATED:")
print(f"    Grid Search:       {n_grid_combinations} combinations (all exhaustive)")
print(f"    Randomized Search: {n_random_iterations} combinations (random sample)")
print(f"    Difference:        Grid evaluates {n_grid_combinations - n_random_iterations} MORE combinations")

print(f"\n(ii) BEST SCORES FOUND:")
print(f"    Grid Search best CV score:       {grid_elastic.best_score_:.4f}")
print(f"    Randomized Search best CV score: {random_elastic.best_score_:.4f}")
score_diff = abs(grid_elastic.best_score_ - random_elastic.best_score_)
print(f"    Difference:                      {score_diff:.4f}")
if grid_elastic.best_score_ > random_elastic.best_score_:
    print(f"    ✓ Grid Search found better score by {score_diff:.4f}")
else:
    print(f"    ✓ Randomized Search found comparable/better score")

print(f"\n(iii) COMPUTATION TIME:")
print(f"    Grid Search time:       {grid_elastic_time:.2f} seconds")
print(f"    Randomized Search time: {random_elastic_time:.2f} seconds")
speedup = grid_elastic_time / random_elastic_time
print(f"    Speedup factor:         {speedup:.2f}x (Randomized is {speedup:.2f}x faster)")

print(f"\n{'='*80}")
print(f"INTERPRETATION")
print(f"{'='*80}")

print(f"""
OVERFITTING ANALYSIS:
- Regularization successfully reduced overfitting from {lr_gap*100:.1f}% to ~{min(gaps.values())*100:.1f}%
- {best_model} achieved the best balance between training and test performance
- Feature selection: Lasso eliminated {20 - np.sum(np.abs(lasso.coef_) > 1e-10)} features

HYPERPARAMETER SEARCH EFFICIENCY:
- Grid Search: Exhaustive, guarantees finding best in grid, but {n_grid_combinations} evaluations
- Randomized Search: Only {n_random_iterations} evaluations, {speedup:.1f}x faster, found comparable results
- For large search spaces, Randomized Search is preferred due to computational efficiency
- Best Ridge alpha: {grid_search.best_params_['alpha']} balances bias-variance tradeoff

RECOMMENDATIONS:
1. Use regularization to combat overfitting (reduced gap from {lr_gap*100:.1f}% to {min(gaps.values())*100:.1f}%)
2. For few hyperparameters (&lt;3) and small ranges: Grid Search
3. For many hyperparameters or continuous ranges: Randomized Search
4. Always use cross-validation to avoid overfitting on validation set
""")

print(f"\n{'='*80}")
print(f"ANALYSIS COMPLETE")
print(f"{'='*80}")


--- GRID SEARCH ---
Parameter grid:
  alpha: [0.001, 0.01, 0.1, 1, 10]
  l1_ratio: [0, 0.25, 0.5, 0.75, 1]
Total combinations: 25
Execution time: 5.88 seconds
Best parameters: {'alpha': 0.1, 'l1_ratio': 1}
Best CV score: 0.9922

--- RANDOMIZED SEARCH ---
Parameter distributions:
  alpha: uniform(0.001, 100)
  l1_ratio: uniform(0, 1)
Random iterations: 25
Execution time: 0.71 seconds
Best parameters: {'alpha': np.float64(2.0594494295802446), 'l1_ratio': np.float64(0.9699098521619943)}
Best CV score: 0.9844

ANSWER (c): Grid Search vs Randomized Search Comparison

(i) NUMBER OF COMBINATIONS EVALUATED:
    Grid Search:       25 combinations (all exhaustive)
    Randomized Search: 25 combinations (random sample)
    Difference:        Grid evaluates 0 MORE combinations

(ii) BEST SCORES FOUND:
    Grid Search best CV score:       0.9922
    Randomized Search best CV score: 0.9844
    Difference:                      0.0078
    ✓ Grid Search found better score by 0.0078

(iii) COMPUTATION 